<!-- AUTOGENERATED-DOCS -->
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Sarbani-Roy/rotating-cylinders-test/main?labpath=notebooks%2Frotating_cylinders.ipynb)

# Rotating Cylinders Benchmark (Taylor-Couette Flow)

## Problem description

This benchmark simulates the flow of a viscous incompressible fluid confined
between two concentric cylinders. The inner cylinder rotates with angular
velocity $\omega_1$ and the outer cylinder with angular velocity $\omega_2$.
For certain parameter combinations the flow is dominated by viscous shear
(Couette flow); above a critical inner-cylinder Reynolds number the flow
becomes unstable and develops the well-known Taylor vortex pattern.

We work in 2D cylindrical coordinates $(r, \theta)$ in the meridional
$(r, z)$ plane. The annulus has inner radius $r_\mathrm{in}$ and outer radius
$r_\mathrm{out}$, with $\eta = r_\mathrm{in}/r_\mathrm{out}$ the radius ratio.

## Governing equations

The incompressible Navier-Stokes equations in cylindrical coordinates read

$$
\begin{aligned}
\rho\left(\frac{\partial \boldsymbol u}{\partial t} + (\boldsymbol u \cdot \nabla)\boldsymbol u\right) &= -\nabla p + \mu \Delta \boldsymbol u, \\
\nabla \cdot \boldsymbol u &= 0,
\end{aligned}
$$

with density $\rho$ and dynamic viscosity $\mu$. The velocity field is
$\boldsymbol u = (u_r, u_\theta, u_z)$ and $p$ is the pressure.

The non-dimensional control parameters are

- The **Reynolds number** of the inner cylinder
  $\mathrm{Re}_1 = \rho\, \omega_1\, r_\mathrm{in}\, (r_\mathrm{out} - r_\mathrm{in}) / \mu$.
- The **radius ratio** $\eta = r_\mathrm{in}/r_\mathrm{out}$.
- The **aspect ratio** $\Gamma = L / (r_\mathrm{out} - r_\mathrm{in})$, where
  $L$ is the axial length of the domain.
- The **outer-cylinder Reynolds number**
  $\mathrm{Re}_2 = \rho\, \omega_2\, r_\mathrm{out}\, (r_\mathrm{out} - r_\mathrm{in}) / \mu$.

## Domain and boundary conditions

The computational domain is the open annulus
$\Omega = \{(r,\theta,z) \mid r_\mathrm{in} < r < r_\mathrm{out},\; 0 < z < L\}$
in a 2D $(r,z)$ cross-section. The boundary conditions are

$$
\begin{aligned}
\boldsymbol u(r=r_\mathrm{in}) &= (0,\; \omega_1 r_\mathrm{in},\; 0), \\
\boldsymbol u(r=r_\mathrm{out}) &= (0,\; \omega_2 r_\mathrm{out},\; 0), \\
\boldsymbol u(z=0) &= \boldsymbol u(z=L) \quad \text{(periodic)}, \\
p &\text{ pinned at one point to remove the null space.}
\end{aligned}
$$

## Analytical reference: circular Couette flow

For steady, axisymmetric, purely azimuthal flow the analytical solution is

$$
u_\theta^\mathrm{Couette}(r) = A r + \frac{B}{r},
$$

with constants

$$
\begin{aligned}
A &= \frac{\omega_2 r_\mathrm{out}^2 - \omega_1 r_\mathrm{in}^2}{r_\mathrm{out}^2 - r_\mathrm{in}^2}, \\
B &= \frac{(\omega_1 - \omega_2) r_\mathrm{in}^2 r_\mathrm{out}^2}{r_\mathrm{out}^2 - r_\mathrm{in}^2}.
\end{aligned}
$$

This is the reference used to compute the relative $L^2$ error of the velocity
field in the postprocessing step.

## Output metrics

- `l2_error_velocity_rel` — relative $L^2$ error of the velocity field with
  respect to the analytical Couette solution.
- `solution_metrics.json` — per-configuration metrics, written by the
  postprocessing script.

## Table of parameters

### Model parameters

| Parameter             | Description                                |
| --------------------- | ------------------------------------------ |
| $r_\mathrm{in}$[m]    | Inner cylinder radius.                     |
| $r_\mathrm{out}$[m]   | Outer cylinder radius.                     |
| $L$[m]                | Axial length of the domain.                |
| $\rho$[kg/m³]         | Fluid density.                             |
| $\mu$[Pa·s]           | Dynamic viscosity.                         |
| $\omega_1$[rad/s]     | Inner-cylinder angular velocity.           |
| $\omega_2$[rad/s]     | Outer-cylinder angular velocity.           |

### Numerical parameters

| Parameter         | Description                                |
| ----------------- | ------------------------------------------ |
| mesh refinement   | Number / size of cells per configuration.  |
| solver tolerances | Linear and non-linear solver tolerances.   |
| time step         | Time step (for transient runs).           |

## Numerical Results

The generated notebook `notebooks/rotating_cylinders.ipynb` is rebuilt by the
`merge-docs-to-notebooks` GitHub Actions workflow and is uploaded as an
artifact on every push to `main`, PR to `main`, and manual dispatch.

test


In [ ]:
"""
plot_convergence.py
-------------------
General convergence plotting script for the rotating-cylinders benchmark.

Place this file at:
    benchmarks/rotating-cylinders/plot_convergence.py

It auto-discovers every solver subfolder that contains a results/ directory,
reads solution_metrics.json and parameters.json from each case, and produces:

  1. One plot per solver  →  <solver>/results/solution_metrics_plot.png
  2. One combined plot    →  rotating-cylinders/convergence_comparison.png
"""
import json
import matplotlib.pyplot as plt
from pathlib import Path
root_dir    = Path(__file__).resolve().parent   # rotating-cylinders/
results_key = "results"                         # expected subfolder name
solvers = sorted(
    d for d in root_dir.iterdir()
    if d.is_dir() and (d / results_key).is_dir()
)
if not solvers:
    print("No solver directories with a 'results/' folder found. Exiting.")
    exit(1)
print(f"Found solvers: {[s.name for s in solvers]}\n")
print("--- Per-solver plots ---")
solver_data = {}   # name → list[dict]  (kept for the combined plot)
for solver_dir in solvers:
    data = load_solver_results(solver_dir)
    if not data:
        print(f"  [skip] no valid cases for {solver_dir.name}")
        continue

    solver_data[solver_dir.name] = data
    save_path = solver_dir / results_key / "solution_metrics_plot.png"
    plot_single_solver(solver_dir.name, data, save_path)
print("\n--- Combined comparison plot ---")
if len(solver_data) < 2:
    print("  Only one solver with data — skipping combined plot.")
else:
    # Use a distinct marker per solver so lines stay distinguishable in B&W too
    markers = ["o", "s", "^", "D", "v", "P", "X", "*"]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

    for ax, error_key, field in zip(
        axes,
        ["pressure_error", "velocity_error"],
        ["Pressure", "Velocity"],
    ):
        for idx, (solver_name, data) in enumerate(solver_data.items()):
            confs  = [d["conf"]          for d in data]
            errors = [d[error_key]       for d in data]
            marker = markers[idx % len(markers)]
            ax.plot(confs, errors, marker=marker, linewidth=1.5, label=solver_name.capitalize())

        ax.set_yscale("log")
        ax.set_xlabel("Configuration")
        ax.set_ylabel(r"Relative $L^2$ Error")
        ax.set_title(f"Convergence Comparison — {field}")
        ax.legend()
        ax.grid(True, which="both", linestyle="-", alpha=0.3)
        plt.setp(ax.get_xticklabels(), rotation=45)

    fig.suptitle("Rotating Cylinders — Solver Convergence Comparison", fontsize=14)
    plt.tight_layout()

    combined_path = root_dir / "convergence_comparison.png"
    fig.savefig(combined_path, dpi=150)
    plt.close(fig)
    print(f"  Saved: {combined_path.relative_to(root_dir)}")
print("\nDone.")


In [ ]:
def load_solver_results(solver_dir: Path) -> list[dict]:
    """Return a list of result dicts sorted by cells_radial (or conf name)."""
    results_dir = solver_dir / results_key
    all_results = []

    for case_dir in results_dir.iterdir():
        if not case_dir.is_dir():
            continue
        metrics_file = case_dir / "solution_metrics.json"
        params_file  = case_dir / "parameters.json"

        if not metrics_file.exists():
            print(f"  [skip] no solution_metrics.json in {solver_dir.name}/{case_dir.name}")
            continue

        metrics = json.loads(metrics_file.read_text())
        params  = json.loads(params_file.read_text()) if params_file.exists() else {}

        all_results.append({
            "conf":         case_dir.name,
            "cells_radial": params.get("grid", {}).get("cells_radial"),
            "pressure_error": metrics.get("l2_error_pressure_rel"),
            "velocity_error": metrics.get("l2_error_velocity_rel"),
        })

    if not all_results:
        return all_results

    # Sort by cells_radial when available, otherwise by conf name
    if all(d["cells_radial"] is not None for d in all_results):
        all_results.sort(key=lambda d: d["cells_radial"])
    else:
        all_results.sort(key=lambda d: d["conf"])

    return all_results


In [ ]:
def plot_single_solver(solver_name: str, data: list[dict], save_path: Path) -> None:
    confs    = [d["conf"]            for d in data]
    p_errors = [d["pressure_error"]  for d in data]
    v_errors = [d["velocity_error"]  for d in data]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(confs, p_errors, marker="o", linewidth=1.5,
            label=r"Relative $L^2$ Error — Pressure")
    ax.plot(confs, v_errors, marker="s", linewidth=1.5,
            label=r"Relative $L^2$ Error — Velocity")

    ax.set_yscale("log")
    ax.set_xlabel("Configuration")
    ax.set_ylabel(r"Relative $L^2$ Error")
    ax.set_title(f"{solver_name.capitalize()} Rotating Cylinders — Convergence Summary")
    ax.legend()
    ax.grid(True, which="both", linestyle="-", alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()

    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Saved: {save_path.relative_to(root_dir)}")
